# A generic branch-and-bound algorithm for l0-regularized problems.

*C. Elvira, T. Guyard, and C. Herzet. Submitted in 2025.*

#### Notebook to reproduce experiments of Sections 6.2 and 6.3

---

These experiments compare the performance of different methods to solver L0-norm problem associated with the reconstruction of signals generated from Bernoulli mixture models. To reproduce it, let first import all the necessary packages and routines.

In [1]:
import numpy as np
from el0ps.utils import compute_lmbd_max
from l0exp.calibration import get_calibration_mixture
from l0exp.dataset import get_dataset_mixture
from l0exp.solver import get_solver, can_handle_instance

Now, let generate the data, that is, a noisy signal `y = Ax + Ɛ`, where:

- `A in R^{m x n}` is a matrix with rows drawn from a multivariate Normal distribution `N(0,K)` with covariance matrix `K in R^{n x n}` defined as `K[i,j] = r^|i-j|`,
- `x in R^{n}` is a sparse signal with `k` non-zero entries evenly spaces with amplitude drawn from a given `distrib_name` distribution with parameters `distrib_params`,
- `Ɛ in R^{m}` is a Gaussian noise with signal-to-noise ratio `s` with respect to `Ax` on average.

In [2]:
mixture_args = {
    'k'             : 10,                           # number of non-zero entries in x
    'm'             : 500,                          # number of rows of A / entries in y
    'n'             : 1000,                         # number of columns of A / entries in x
    'r'             : 0.9,                          # correlation matrix coefficient
    's'             : 10.0,                         # signal-to-noise ratio
    'distrib_name'  : "gaussian",                   # name of the distribution to draw the non-zero entries of x
    'distrib_args'  : {"scale": 1.0},               # params of the distribution to draw the non-zero entries of x
    'seed'          : None,                         # optional int seed for reproducibility (no seed set if None)
}


A, y, x = get_dataset_mixture(**mixture_args)
print(f"A shape: {A.shape}")
print(f"y shape: {y.shape}")
print(f"x shape: {x.shape}")
print(f"nnz    : {np.count_nonzero(x)}")
print(f"snr    : {np.linalg.norm(A @ x)**2 / np.linalg.norm(y - A @ x)**2:.2f}")

A shape: (500, 1000)
y shape: (500,)
x shape: (1000,)
nnz    : 10
snr    : 10.38


The different values for `distrib_name` and `distrib_params` that we used in the experiment are as follows:

| Distribution   | `distrib_name`            | `distrib_params`                 |
|----------------|---------------------------|----------------------------------|
| Uniform        | `"uniform"`               | `{"low": -1.0, "high": 1.0}`     |
| Normal         | `"gaussian"`              | `{"scale": 1.0}`                 |
| Laplace        | `"laplace"`               | `{"scale": 1.0}`                 |
| Exponential    | `"exponential"`           | `{"scale": 1.0}`                 |
| Half-Normal    | `"halfgaussian"`          | `{"scale": 1.0}`                 |
| Gauss-Laplace  | `"gausslaplace"`          | `{"scale1": 1.0, "scale2": 1.0}` |

The Normal, Laplace, Half-Normal, and Gauss-Laplace distributions need to be centered.

From the mixture model parameters, we can calibrate appropriate data-fidelity and penalty functions, as well as the value of the regularization parameter `lmbd`.

In [3]:
datafit, penalty, lmbd = get_calibration_mixture(A, y, x, **mixture_args)
print(f"datafit   : {datafit}")
print(f"penalty   : {penalty}")
print(f"lambda    : {lmbd}")
print(f"lambda_max: {compute_lmbd_max(datafit, penalty, A)}")

datafit   : Leastsquares
penalty   : L2norm
lambda    : 0.37009656946970176
lambda_max: 23.67246724642128


Finally, we can define the different methods to compare and their parameters. Here is the setup used in our experiments.

- For the experiment in **Section 6.2**, this solving procedure has been repeated `100` times to plot performance profiles with the number of instances solved within a given time limit.
- For the experiment in **Section 6.3**, this solving procedure has been performed for the **Uniform** density function with different parameters in `mixture_args` and repeated `10` times to perform the sensitivity analysis.

Solvers that cannot handle the considered instance are automatically skipped.

In [4]:
# Solvers to use (comment out those you don't want to run or that are not installed)
solver_types = [
    "el0ps",
    "l0bnb",
    "mimosa",
    "gurobi",
    "mosek",
    "oa"
]

solver_args = {
    "time_limit"    : 60,     # time limit in seconds for solvers
    "relative_gap"  : 1.e-8,  # relative optimality gap on the objective value
    "verbose"       : False,  # verbosity toggle for solvers
}


for solver_type in solver_types:

    try:    
        solver = get_solver(solver_type, solver_args)
        if can_handle_instance(solver, datafit, penalty):
            print(f"Running {solver_type}...")
            result = solver.solve(datafit, penalty, A, lmbd)
            print(f"  status    : {result.status}")
            print(f"  objective : {result.objective_value:.4f}")
            print(f"  non-zeros : {np.count_nonzero(result.x)}")
            print(f"  solve time: {result.solve_time:.4f}")
        else:
            print(f"Skipping {solver_type}: cannot handle this instance")
    except Exception as e:
        print(f"Solver {solver_type} failed with error: {e}")
    print()

Running el0ps...
  status    : optimal
  objective : 3.0815
  non-zeros : 4
  solve time: 2.4131

Running l0bnb...


KeyboardInterrupt: 